# Experimento 3 - Sensibilidade ao limiar minimo

Objetivo: avaliar o impacto de `qkd_min_bits_threshold`, parametro explicito do enlace usado pelas politicas `threshold` e `hybrid`.

Contexto: o enlace inicia com `qkd_min_bits_threshold = 128`, e esse valor aparece no estado monitorado do link.

Este experimento varre os limiares:
- 32
- 64
- 128
- 256

e executa para as politicas:
- threshold
- hybrid

In [1]:
import sys
import random
import pandas as pd

sys.path.insert(0, '/home/esdras/Wqnets/QuantumNet')

from quantumnet.topology import Network

print('Imports carregados com sucesso')

Imports carregados com sucesso


## Configuracao

Carga fixa (intermediaria), para isolar o efeito do limiar:
- topologia: Linha(4)
- enlaces avaliados: (0,1), (1,2), (2,3)
- requisicoes por enlace: 8
- tamanhos possiveis por requisicao (bits): 24, 32, 48, 64

In [2]:
policies = ['threshold', 'hybrid']
threshold_values = [32, 64, 128, 256]
link_pairs = [(0, 1), (1, 2), (2, 3)]

requests_per_link = 8
request_bit_options = [24, 32, 48, 64]
trials_per_setting = 10

print('Politicas:', policies)
print('Limiares:', threshold_values)
print('Repeticoes por configuracao:', trials_per_setting)

Politicas: ['threshold', 'hybrid']
Limiares: [32, 64, 128, 256]
Repeticoes por configuracao: 10


## Execucao do experimento

Metricas observadas por repeticao:
- `served_requests` e `denied_requests`
- `total_generated_bits` e `total_consumed_bits`
- `replenishment_events`
- `bits_available` final

Indicador adicional de escassez instantanea:
- `instant_shortage_events`: contagem de vezes em que, imediatamente antes do pedido, `bits_available < requested_bits`.

In [3]:
def run_single_trial(policy: str, min_threshold: int, trial_id: int, base_seed: int = 3030) -> dict:
    rng = random.Random(base_seed + trial_id * 1000 + min_threshold * 10 + (1 if policy == 'hybrid' else 0))

    net = Network()
    net.set_ready_topology('Linha', 4)
    net.controller.set_policy(policy)

    for alice_id, bob_id in link_pairs:
        net.controller.set_minimum_stock(alice_id, bob_id, min_threshold)

    instant_shortage_events = 0

    for alice_id, bob_id in link_pairs:
        for _ in range(requests_per_link):
            requested_bits = rng.choice(request_bit_options)

            state_before = net.get_qkd_link_state(alice_id, bob_id)
            if int(state_before['bits_available']) < int(requested_bits):
                instant_shortage_events += 1

            net.controller.handle_key_request(alice_id, bob_id, requested_bits)

    totals = {
        'total_generated_bits': 0,
        'total_consumed_bits': 0,
        'served_requests': 0,
        'denied_requests': 0,
        'replenishment_events': 0,
        'bits_available': 0,
    }

    for alice_id, bob_id in link_pairs:
        state = net.get_qkd_link_state(alice_id, bob_id)
        totals['total_generated_bits'] += int(state['total_generated_bits'])
        totals['total_consumed_bits'] += int(state['total_consumed_bits'])
        totals['served_requests'] += int(state['served_requests'])
        totals['denied_requests'] += int(state['denied_requests'])
        totals['replenishment_events'] += int(state['replenishment_events'])
        totals['bits_available'] += int(state['bits_available'])

    served = totals['served_requests']
    denied = totals['denied_requests']
    generated = totals['total_generated_bits']
    consumed = totals['total_consumed_bits']

    service_rate = served / (served + denied) if (served + denied) > 0 else 0.0
    efficiency = consumed / generated if generated > 0 else 0.0

    return {
        'policy': policy,
        'min_threshold': min_threshold,
        'trial': trial_id,
        **totals,
        'service_rate': service_rate,
        'efficiency': efficiency,
        'instant_shortage_events': instant_shortage_events,
    }

rows = []
for policy in policies:
    for min_threshold in threshold_values:
        for trial in range(1, trials_per_setting + 1):
            rows.append(run_single_trial(policy, min_threshold, trial))

results_df = pd.DataFrame(rows)
results_df.head()

,policy,min_threshold,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency,instant_shortage_events
0,threshold,32,1,496,488,13,11,13,8,0.541667,0.983871,24
1,threshold,32,2,432,376,12,12,12,56,0.500000,0.870370,24
2,threshold,32,3,496,472,11,13,11,24,0.458333,0.951613,24
3,threshold,32,4,512,480,13,11,13,32,0.541667,0.937500,24
4,threshold,32,5,496,456,13,11,13,40,0.541667,0.919355,24


## Resultado detalhado por repeticao

In [4]:
display(results_df.sort_values(['policy', 'min_threshold', 'trial']).reset_index(drop=True))

,policy,min_threshold,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency,instant_shortage_events
0,hybrid,32,1,920,912,22,2,30,8,0.916667,0.991304,24
1,hybrid,32,2,776,768,20,4,28,8,0.833333,0.989691,24
2,hybrid,32,3,936,928,22,2,34,8,0.916667,0.991453,24
3,hybrid,32,4,864,864,21,3,32,0,0.875000,1.000000,23
4,hybrid,32,5,696,664,20,4,23,32,0.833333,0.954023,24
...,...,...,...,...,...,...,...,...,...,...,...,...
75,threshold,256,6,0,0,0,24,0,0,0.000000,0.000000,24
76,threshold,256,7,0,0,0,24,0,0,0.000000,0.000000,24
77,threshold,256,8,256,256,6,18,1,0,0.250000,1.000000,19
78,threshold,256,9,0,0,0,24,0,0,0.000000,0.000000,24


## Resumo agregado

Leitura principal:
- taxa de atendimento
- eficiencia de uso
- pressao sobre o sistema (reposicoes)
- escassez residual
- risco de falta instantanea

In [5]:
summary_df = (
    results_df
    .groupby(['policy', 'min_threshold'], as_index=False)
    .agg(
        service_rate_mean=('service_rate', 'mean'),
        service_rate_std=('service_rate', 'std'),
        efficiency_mean=('efficiency', 'mean'),
        replenishment_events_mean=('replenishment_events', 'mean'),
        bits_available_mean=('bits_available', 'mean'),
        instant_shortage_events_mean=('instant_shortage_events', 'mean'),
        denied_requests_mean=('denied_requests', 'mean'),
        served_requests_mean=('served_requests', 'mean'),
    )
    .sort_values(['policy', 'min_threshold'])
    .reset_index(drop=True)
)

display(summary_df)

,policy,min_threshold,service_rate_mean,service_rate_std,efficiency_mean,replenishment_events_mean,bits_available_mean,instant_shortage_events_mean,denied_requests_mean,served_requests_mean
0,hybrid,32,0.845833,0.076199,0.988807,29.0,8.8,23.7,3.7,20.3
1,hybrid,64,0.820833,0.044140,0.964399,17.6,31.2,21.5,4.3,19.7
2,hybrid,128,0.666667,0.070820,1.000000,14.5,0.0,22.4,8.0,16.0
3,hybrid,256,0.654167,0.122742,1.000000,14.9,0.0,23.2,8.3,15.7
4,threshold,32,0.550000,0.078075,0.932455,13.2,34.4,24.0,10.8,13.2
5,threshold,64,0.270833,0.111544,0.856667,4.4,40.8,21.9,17.5,6.5
6,threshold,128,0.087500,0.074665,0.571875,0.8,16.8,22.7,21.9,2.1
7,threshold,256,0.041667,0.090010,0.178125,0.2,5.6,23.2,23.0,1.0


## Leitura interpretativa automatica

Padroes esperados:
- limiar maior tende a aumentar `replenishment_events`
- limiar menor tende a aumentar risco de falta instantanea
- `hybrid` tende a ficar mais equilibrado em carga intermediaria por combinar manutencao de estoque e reposicao por necessidade

In [6]:
print('Resumo por politica:')
for policy in policies:
    subset = summary_df[summary_df['policy'] == policy].sort_values('min_threshold')

    print(f'\nPolitica: {policy}')
    for _, row in subset.iterrows():
        print(
            f"  limiar={int(row['min_threshold']):3d} | "
            f"atendimento={row['service_rate_mean']:.3f} | "
            f"eficiencia={row['efficiency_mean']:.3f} | "
            f"reposicoes={row['replenishment_events_mean']:.1f} | "
            f"falta_instantanea={row['instant_shortage_events_mean']:.1f} | "
            f"buffer_final={row['bits_available_mean']:.1f}"
        )

print('\nComparacao direta threshold vs hybrid por limiar:')
pivot_compare = summary_df.pivot(index='min_threshold', columns='policy', values='service_rate_mean')
display(pivot_compare)

Resumo por politica:

Politica: threshold
  limiar= 32 | atendimento=0.550 | eficiencia=0.932 | reposicoes=13.2 | falta_instantanea=24.0 | buffer_final=34.4
  limiar= 64 | atendimento=0.271 | eficiencia=0.857 | reposicoes=4.4 | falta_instantanea=21.9 | buffer_final=40.8
  limiar=128 | atendimento=0.087 | eficiencia=0.572 | reposicoes=0.8 | falta_instantanea=22.7 | buffer_final=16.8
  limiar=256 | atendimento=0.042 | eficiencia=0.178 | reposicoes=0.2 | falta_instantanea=23.2 | buffer_final=5.6

Politica: hybrid
  limiar= 32 | atendimento=0.846 | eficiencia=0.989 | reposicoes=29.0 | falta_instantanea=23.7 | buffer_final=8.8
  limiar= 64 | atendimento=0.821 | eficiencia=0.964 | reposicoes=17.6 | falta_instantanea=21.5 | buffer_final=31.2
  limiar=128 | atendimento=0.667 | eficiencia=1.000 | reposicoes=14.5 | falta_instantanea=22.4 | buffer_final=0.0
  limiar=256 | atendimento=0.654 | eficiencia=1.000 | reposicoes=14.9 | falta_instantanea=23.2 | buffer_final=0.0

Comparacao direta threshol

policy,hybrid,threshold
min_threshold,,
32,0.845833,0.550000
64,0.820833,0.270833
128,0.666667,0.087500
256,0.654167,0.041667


## O que este experimento responde

Mostra a sensibilidade das politicas `threshold` e `hybrid` ao parametro `qkd_min_bits_threshold` e em que regime o ajuste de limiar melhora estabilidade (atendimento) ao custo de maior reposicao.

Tambem evidencia o trade-off entre manter estoque (menos falta instantanea) e aumentar pressao de reposicao no sistema.